# 🔍 Similarity Search

**Day 5 — Notebook 3 of 4 | Estimated Time: 30 minutes**

---

## What You'll Learn
- Cosine similarity and dot product — the maths behind semantic search
- Building a brute-force similarity search from scratch
- Ranking results by relevance
- Comparing keyword search vs semantic search
- Handling edge cases: short queries, ambiguous terms

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0" numpy

In [ ]:
import sys, os
import numpy as np

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
EMBEDDING_MODEL = "text-embedding-004"
print("✅ Ready!")

---

## 1. The Mathematics of Similarity

There are two main ways to measure how similar two vectors are:

### Cosine Similarity
Measures the **angle** between two vectors, regardless of their magnitude.

```
cosine_similarity(A, B) = (A · B) / (‖A‖ × ‖B‖)

Range: -1 to +1
  1.0  = identical direction (maximum similarity)
  0.0  = orthogonal (no similarity)
 -1.0  = opposite directions
```

### Dot Product
The raw dot product `A · B`. When vectors are **normalised** (L2 norm = 1), dot product equals cosine similarity.

Most embedding APIs, including Gemini's, return normalised vectors — so dot product and cosine similarity are equivalent.

In [ ]:
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def dot_product(a: np.ndarray, b: np.ndarray) -> float:
    """Raw dot product."""
    return float(np.dot(a, b))


# Verify that Gemini embeddings are normalised (L2 norm ≈ 1.0)
result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=["Hello world", "The sky is blue"],
)
v1 = np.array(result.embeddings[0].values)
v2 = np.array(result.embeddings[1].values)

print(f"L2 norm of v1: {np.linalg.norm(v1):.6f}")
print(f"L2 norm of v2: {np.linalg.norm(v2):.6f}")
print(f"Cosine similarity: {cosine_similarity(v1, v2):.6f}")
print(f"Dot product:       {dot_product(v1, v2):.6f}")
print("\nSince norms ≈ 1.0, cosine similarity == dot product ✅")

---

## 2. Building a Simple Search Index

A **search index** is just a collection of (text, embedding) pairs. To search it, embed the query and find the closest embeddings.

In [ ]:
class SimpleSearchIndex:
    """A brute-force semantic search index."""
    
    def __init__(self, model: str = EMBEDDING_MODEL):
        self.model = model
        self.documents = []   # List of (id, text, metadata)
        self.vectors = []     # Corresponding embedding vectors
    
    def add_documents(self, docs: list[dict]):
        """Add documents to the index. Each doc needs 'id' and 'text' keys."""
        texts = [d["text"] for d in docs]
        
        # Batch embed all new documents
        result = client.models.embed_content(
            model=self.model,
            contents=texts,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
        )
        
        for doc, embedding in zip(docs, result.embeddings):
            self.documents.append(doc)
            self.vectors.append(np.array(embedding.values))
        
        print(f"✅ Added {len(docs)} documents. Index size: {len(self.documents)}")
    
    def search(self, query: str, top_k: int = 3) -> list[dict]:
        """Search the index. Returns top_k most relevant documents."""
        # Embed the query
        result = client.models.embed_content(
            model=self.model,
            contents=query,
            config=types.EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
        )
        query_vec = np.array(result.embeddings[0].values)
        
        # Compute similarities against all documents
        matrix = np.array(self.vectors)  # shape: (n_docs, 768)
        scores = matrix @ query_vec      # dot product = cosine similarity (normalised vecs)
        
        # Rank by score descending
        ranked_indices = np.argsort(scores)[::-1][:top_k]
        
        return [
            {**self.documents[i], "score": float(scores[i])}
            for i in ranked_indices
        ]


print("SimpleSearchIndex class ready.")

In [ ]:
# Build the index with a set of knowledge base documents
knowledge_base = [
    {"id": "1", "text": "Python is a high-level programming language known for its simple syntax and readability.", "category": "tech"},
    {"id": "2", "text": "Machine learning algorithms learn patterns from data without explicit programming.", "category": "tech"},
    {"id": "3", "text": "The mitochondria is the powerhouse of the cell, responsible for producing ATP.", "category": "biology"},
    {"id": "4", "text": "Photosynthesis converts sunlight, water, and CO2 into glucose and oxygen.", "category": "biology"},
    {"id": "5", "text": "The French Revolution began in 1789 and fundamentally transformed French society.", "category": "history"},
    {"id": "6", "text": "World War II ended in 1945 with the surrender of Nazi Germany and Imperial Japan.", "category": "history"},
    {"id": "7", "text": "Neural networks are computational models inspired by the structure of the human brain.", "category": "tech"},
    {"id": "8", "text": "DNA carries genetic instructions for the development of living organisms.", "category": "biology"},
    {"id": "9", "text": "The Roman Empire at its peak controlled much of Europe, North Africa, and Western Asia.", "category": "history"},
    {"id": "10", "text": "APIs (Application Programming Interfaces) allow software systems to communicate.", "category": "tech"},
]

index = SimpleSearchIndex()
index.add_documents(knowledge_base)

In [ ]:
# Run some searches!
queries = [
    "How do computers learn from data?",
    "What happened during the Second World War?",
    "How do cells produce energy?",
    "How can one program connect to another?",
]

for query in queries:
    print(f"🔍 Query: '{query}'")
    results = index.search(query, top_k=3)
    for rank, r in enumerate(results, 1):
        print(f"  {rank}. [{r['score']:.3f}] ({r['category']}) {r['text'][:80]}")
    print()

---

## 3. Keyword Search vs. Semantic Search

A critical comparison: **keyword search** matches exact words, **semantic search** matches meaning. They fail in different ways.

In [ ]:
def keyword_search(query: str, documents: list[dict], top_k: int = 3) -> list[dict]:
    """Simple TF-based keyword search (counts query word matches)."""
    query_words = set(query.lower().split())
    scored = []
    for doc in documents:
        doc_words = set(doc["text"].lower().split())
        overlap = len(query_words & doc_words)
        scored.append({**doc, "score": overlap})
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]


# Queries designed to expose keyword search weaknesses
test_queries = [
    "How does the cell make energy?",           # 'energy' → mitochondria (semantic wins)
    "ancient empire in europe",                  # No exact matches — semantic should win
    "Python machine learning library",           # Multiple keywords — keyword may win
]

for query in test_queries:
    print(f"Query: '{query}'")
    
    kw_results = keyword_search(query, knowledge_base, top_k=2)
    sem_results = index.search(query, top_k=2)
    
    print("  Keyword Search:")
    for r in kw_results:
        print(f"    [{r['score']} matches] {r['text'][:70]}")
    
    print("  Semantic Search:")
    for r in sem_results:
        print(f"    [{r['score']:.3f}] {r['text'][:70]}")
    print()

---

## 4. The Similarity Score Threshold

Not all search results are good results. Use a **threshold** to filter out low-relevance matches.

In [ ]:
def search_with_threshold(index: SimpleSearchIndex, query: str, threshold: float = 0.6, max_results: int = 5):
    """Return results only above a minimum similarity threshold."""
    all_results = index.search(query, top_k=max_results)
    filtered = [r for r in all_results if r["score"] >= threshold]
    return filtered


# Test with relevant and irrelevant queries
queries = [
    "How does DNA work?",                          # Relevant
    "What is the weather like in Paris today?",    # Irrelevant — no good match exists
    "Explain machine learning neural networks",   # Relevant
]

for query in queries:
    results = search_with_threshold(index, query, threshold=0.6)
    print(f"Query: '{query}'")
    if results:
        for r in results:
            print(f"  ✅ [{r['score']:.3f}] {r['text'][:70]}")
    else:
        print("  ❌ No results above threshold — query may be out of scope.")
    print()

---

## 5. Building a Similarity Matrix

A **similarity matrix** shows the pairwise similarity between all items in a set — useful for understanding the structure of your data.

In [ ]:
import matplotlib.pyplot as plt

# Use a small set for clear visualisation
sample_texts = [
    "Python programming",
    "Machine learning models",
    "Neural networks AI",
    "Cell biology",
    "DNA genetics",
    "Roman history",
]

result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=sample_texts,
)
vecs = np.array([e.values for e in result.embeddings])

# Compute full similarity matrix
sim_matrix = vecs @ vecs.T  # All pairs at once (works because vecs are normalised)

# Plot as a heatmap
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine Similarity")

labels = [t[:20] for t in sample_texts]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right")
ax.set_yticklabels(labels)

# Annotate cells
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Pairwise Semantic Similarity Matrix")
plt.tight_layout()
plt.show()

---

## 🏋️ Exercise 1: Improve the Search Index

Add a **metadata filter** to the search index so users can restrict results to a specific category.

In [ ]:
# Exercise 1: Add metadata filtering to search
# TODO: Modify the search method (or build a wrapper) so that:
#   search(query, top_k=3, filter_category="biology")
#   ...only returns results where doc["category"] == "biology"

def filtered_search(index: SimpleSearchIndex, query: str, top_k: int = 3, category: str = None) -> list[dict]:
    """Search with optional category filter."""
    # TODO: Implement this
    # Hint: Get more results than needed, then filter by category before returning top_k
    pass


# Test:
# results = filtered_search(index, "How do living things work?", top_k=2, category="biology")
# for r in results:
#     print(f"[{r['score']:.3f}] ({r['category']}) {r['text'][:70]}")

---

## 🏋️ Exercise 2: Find Near-Duplicates

Use the similarity matrix to find documents in the knowledge base that are suspiciously similar (potential duplicates or near-duplicates).

In [ ]:
# Exercise 2: Detect near-duplicate documents
# This corpus contains some near-duplicates — find them!
corpus = [
    "Python is an interpreted, high-level programming language.",
    "Python is a high-level, interpreted programming language.",  # Near-duplicate!
    "Machine learning uses data to train predictive models.",
    "The human heart pumps blood throughout the body.",
    "Algorithms learn patterns from data in machine learning.",   # Near-duplicate!
    "The heart is a muscle that circulates blood through the body.",  # Near-duplicate!
    "JavaScript is used for web development.",
]

# TODO:
# 1. Embed all documents in corpus
# 2. Compute pairwise similarity matrix (excluding self-similarity on the diagonal)
# 3. Print all pairs with similarity > 0.9 (these are likely near-duplicates)

# Your code here

---

## Key Takeaways

| Concept | Summary |
|---------|----------|
| **Cosine similarity** | Angle-based similarity; 1.0 = identical, 0.0 = unrelated |
| **Dot product shortcut** | When vectors are normalised (as Gemini's are), dot product = cosine similarity |
| **Brute-force search** | `matrix @ query_vector` computes all similarities in one vectorised operation |
| **Threshold filtering** | Only return results above a minimum score to avoid irrelevant matches |
| **Semantic vs keyword** | Semantic search understands meaning; keyword search matches exact words |

---

## 📚 Further Reading

| Resource | Type | Link |
|----------|------|------|
| Cosine Similarity | Wikipedia | [en.wikipedia.org/wiki/Cosine_similarity](https://en.wikipedia.org/wiki/Cosine_similarity) |
| Approximate Nearest Neighbours | Blog | [ann-benchmarks.com](https://ann-benchmarks.com/) |

---

**Next up:** [04_Semantic_Search_Application.ipynb](./04_Semantic_Search_Application.ipynb)